In [2]:
from pathlib import Path
import pandas as pd

data_dir = Path(".").absolute() / "eval-all" / "data-nesy-raw"


In [6]:
all_data = {}

for folder in data_dir.iterdir():
    if not folder.is_dir():
        print(f"Skipping {folder}, not a directory")
        continue

    model_name = folder.name.rsplit("_", 1)[1]
    replace_map = {
        "llama": "NeSy Llama 3.1 8B",
        "qwen": "NeSy Qwen3 8B",
    }
    model_name = replace_map.get(model_name, model_name)
    print(f"Processing {model_name}")

    # We have the model with and without ground truth (GT) actions, and the action encoder (AE)
    dfs_with_gt = []
    dfs_with_ae = []

    for file in folder.glob("*.jsonl"):
        df = pd.read_json(file, lines=True)
        if "_gt_" in file.name:
            dfs_with_gt.append(df.rename(columns={"pred_answer": "prediction_text"}))
        elif "_nogt_" in file.name:
            dfs_with_ae.append(df.rename(columns={"pred_answer": "prediction_text"}))
        else:
            print(f"Skipping {file.name}, not a GT or AE file")
            continue

    all_data[model_name + " GT"] = pd.concat(dfs_with_gt, ignore_index=True)
    all_data[model_name + " AE"] = pd.concat(dfs_with_ae, ignore_index=True)

print()
print(f"Total models processed: {len(all_data)}")

Processing NeSy Qwen3 8B
Processing NeSy Llama 3.1 8B
Skipping /workspaces/ts-qa/eval-all/data-nesy-raw/combine.ipynb, not a directory

Total models processed: 4


In [7]:
# Save as a CSV file, one per model
for model_name, df in all_data.items():
    df.to_csv(
        data_dir.parent / "data" / "4.4" / f"{model_name}_full_test.csv", index=False
    )
